In [1]:
from compmempy.models.memorysearch import Base_CMR
from compmempy.parameters import Parameters
import numpy as np

full_parameters = Parameters('D:/data/base_cmr_parameters.json')
parameters = full_parameters.fixed
item_count = 16

items = np.eye(item_count)
numba_model = Base_CMR(items, item_count, parameters)

numba_model.mfc.activations(items[0])

array([0.        , 0.99999938, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        ])

In [2]:
from jaxcmr.memorysearch import BaseCMR
from jaxcmr.memory import probe
import json
import jax
jax.config.update('jax_enable_x64', True)

with open('D:/data/base_cmr_parameters.json') as f:
    full_parameters = json.load(f)
parameters = full_parameters['fixed']
item_count = 16

jax_model = BaseCMR.create(item_count, parameters)

choice = 1
item_index = choice - 1
encoded_item = jax_model.items[item_index]
context_input = probe(jax_model.mfc, encoded_item)

context_input

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


Array([0.        , 0.89544394, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        ], dtype=float64)

## Scratch

In [7]:
new_context = integrate(jax_model.context, context_input, jax_model.encoding_drift_rate)
new_context.state

Array([0.5978168 , 0.71781719, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        ], dtype=float64)

In [4]:
jax_model = BaseCMR.create(item_count, parameters)
jax_model = experience(jax_model, 1)

print('Pre-retrieval')
print(jax_model.context.state)
print()

print(jax_model.mfc.state)

assert(np.allclose(jax_model.context.state, numba_model.context.state()))
assert(np.allclose(jax_model.mfc.state, numba_model.mfc.memory))

Pre-retrieval
[0.5978168  0.71781719 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.        ]

[[0.06250537 0.97049608 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.89544394 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.89544394 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.89544394 0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.      

AssertionError: 